In [8]:
pip install pandas numpy scikit-learn xgboost lightgbm matplotlib

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 2.1 MB/s eta 0:00:04
   --- ------------------------------------ 0.8/8.1 MB 2.0 MB/s eta 0:00:04
   ------ --------------------------------- 1.3/8.1 MB 2.0 MB/s eta 0:00:04
   --------- ------------------------------ 1.8/8.1 MB 2.1 MB/s eta 0:00:03
   ----------- ---------------------------- 2.4/8.1 MB 2.1 MB/s eta 0:00:03
   ------------ --------------------------- 2.6/8.1 MB 2.1 MB/s eta 0:00:03
   --------------- ------------------------ 3.1/8.1 MB 2.0 MB/s eta 0:00:03
   ---------------- ----------------------- 3.4/8.1 MB 2.0 MB/s eta 0:00:03
   ------------------- -------------------- 3.9/8.1 MB 2.0 MB/s eta 0:00:03
   -------------------- ----------------


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
pip install sklearn

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-pypi-packag

| Attack Probability | Meaning                    |
| ------------------ | -------------------------- |
| 0.0000             | Extremely confident Normal |
| 0.25               | Slightly suspicious        |
| 0.50               | Uncertain                  |
| 0.75               | Likely Attack              |
| 0.99               | Almost definitely Attack   |

Logistic Regression : sigmoid function <br>
Random Forest : proportion of trees voting for that class <br>
LDA : Gaussian distribution assumption<br>
LigtGBM / XGBoost : sigmoid of the summation of tree outputs

In [ ]:
# =====================================================
# UNSW-NB15 Binary Classification WITH Calibration
# =====================================================

import pandas as pd
import numpy as np
import time
import random

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# =====================================================
# 1. Load Dataset
# =====================================================

train_df = pd.read_csv("UNSW_NB15_training-set.csv")
test_df = pd.read_csv("UNSW_NB15_testing-set.csv")

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)


# =====================================================
# 2. Prepare Features & Labels
# =====================================================

y_train = train_df["label"]
y_test = test_df["label"]

X_train = train_df.drop(["label"], axis=1)
X_test = test_df.drop(["label"], axis=1)

if "attack_cat" in X_train.columns:
    X_train = X_train.drop(["attack_cat"], axis=1)
    X_test = X_test.drop(["attack_cat"], axis=1)


# =====================================================
# 3. Preprocessing (Dense Output)
# =====================================================

categorical_cols = X_train.select_dtypes(include=["object"]).columns
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)


# =====================================================
# 4. Define Base Models
# =====================================================

base_models = {
    "XGBoost": XGBClassifier(eval_metric="logloss"),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "LDA": LinearDiscriminantAnalysis(),
    "LightGBM": LGBMClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=2000)
}

results = {}


# =====================================================
# 5. Train + Calibrate Models
# =====================================================

for name, base_model in base_models.items():

    print(f"\nTraining + Calibrating {name}...")

    # Wrap model with calibration
    calibrated_model = CalibratedClassifierCV(
        base_model,
        method="sigmoid",  # Platt scaling
        cv=5
    )

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("classifier", calibrated_model)
    ])

    start_train = time.time()
    pipeline.fit(X_train, y_train)
    end_train = time.time()

    train_time = end_train - start_train

    results[name] = {
        "model": pipeline,
        "train_time": train_time
    }

    print(f"{name} Training + Calibration Time: {train_time:.4f} seconds")


# =====================================================
# 6. Random Detection from Test Set
# =====================================================

random_index = random.randint(0, len(X_test) - 1)
sample = X_test.iloc[[random_index]]
true_label = y_test.iloc[random_index]

print("\n=======================================")
print("Random Test Record Index:", random_index)
print("Actual Label:", "Attack" if true_label == 1 else "Normal")
print("=======================================")

print("\n--- Calibrated Detection Results ---")

for name, info in results.items():

    model = info["model"]

    start_detect = time.time()
    prediction = model.predict(sample)[0]
    probs = model.predict_proba(sample)[0]
    end_detect = time.time()

    detect_time = end_detect - start_detect

    print(f"\n{name}")
    print("Predicted:", "Attack" if prediction == 1 else "Normal")
    print(f"Normal Probability: {probs[0]:.6f}")
    print(f"Attack Probability: {probs[1]:.6f}")
    print(f"Detection Time: {detect_time:.6f} seconds")
    print(f"Training + Calibration Time: {info['train_time']:.4f} seconds")

Training shape: (175341, 45)
Testing shape: (82332, 45)

Training + Calibrating XGBoost...


C:\Users\Kruthika\AppData\Local\Temp\ipykernel_13728\3641417628.py:53: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns


XGBoost Training + Calibration Time: 27.7280 seconds

Training + Calibrating Random Forest...
Random Forest Training + Calibration Time: 98.8160 seconds

Training + Calibrating LDA...
LDA Training + Calibration Time: 15.8339 seconds

Training + Calibrating LightGBM...
[LightGBM] [Info] Number of positive: 95472, number of negative: 44800
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016664 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6444
[LightGBM] [Info] Number of data points in the train set: 140272, number of used features: 186
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680621 -> initscore=0.756625
[LightGBM] [Info] Start training from score 0.756625


c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 95473, number of negative: 44800
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017676 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6432
[LightGBM] [Info] Number of data points in the train set: 140273, number of used features: 186
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680623 -> initscore=0.756635
[LightGBM] [Info] Start training from score 0.756635


...

c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 95473, number of negative: 44800
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017327 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6468
[LightGBM] [Info] Number of data points in the train set: 140273, number of used features: 187
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680623 -> initscore=0.756635
[LightGBM] [Info] Start training from score 0.756635
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Logistic Regression Training + Calibration Time: 22.0542 seconds

Random Test Record Index: 35646
Actual Label: Normal

--- Calibrated Detection Results ---

XGBoost
Predicted: Normal
Normal Probability: 0.870266
Attack Probability: 0.129734
Detection Time: 0.028217 seconds
Training + Calibration Time: 27.7280 seconds

Random Forest
Predicted: Normal
Normal Probability: 0.981405
Attack Probability: 0.018595
Detection Time: 0.166495 seconds
Training + Calibration Time: 98.8160 seconds

LDA
Predicted: Normal
Normal Probability: 0.998330
Attack Probability: 0.001670
Detection Time: 0.019888 seconds
Training + Calibration Time: 15.8339 seconds

LightGBM
Predicted: Normal
Normal Probability: 0.852569
Attack Probability: 0.147431
Detection Time: 0.043473 seconds
Training + Calibration Time: 7.3966 seconds

Logistic Regression
Predicted: Normal
Normal Probability: 0.999998
Attack Probability: 0.000002
Detection Time: 0.030010 seconds
Training + Calibration Time: 22.0542 seconds


c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\valida

In [1]:
# ====== 0. Force non-GUI backend ======
import matplotlib
matplotlib.use('Agg')

# ====== 1. Imports ======
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import os

# Models
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


# ====== 2. Load Data ======
train_df = pd.read_csv("UNSW_NB15_grouped_training_class.csv")
test_df  = pd.read_csv("UNSW_NB15_grouped_testing_class.csv")

target_col = "4_class"

# Align test columns with train
test_df = test_df[train_df.columns]


# ====== 3. Features & Labels ======
X_train = train_df.drop(columns=["id", "attack_cat", "label", target_col])
y_train = train_df[target_col]

X_test  = test_df.drop(columns=["id", "attack_cat", "label", target_col])
y_test  = test_df[target_col]


# ====== 4. Identify Numeric vs Categorical ======
categorical_cols = ["proto", "service", "state"]
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]


# ====== 5. Preprocessing ======
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
])


# ====== 6. Encode Target ======
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test  = le.transform(y_test)


# ====== 7. Models ======
models = {
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=len(le.classes_),
        eval_metric="mlogloss",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=500,
        class_weight="balanced"
    ),
    "LDA": LinearDiscriminantAnalysis()
}


# ====== 8. Create Output Folder ======
os.makedirs("feature_importance_plots_main", exist_ok=True)


# ====== 9. Feature Importance Plot Function ======
def plot_feature_importance(feat_names, importance, title, filename):

    indices = np.argsort(importance)[::-1]
    sorted_feat_names = np.array(feat_names)[indices]
    sorted_importance = importance[indices]

    plt.figure(figsize=(12, max(8, len(feat_names) * 0.3)))
    plt.barh(range(len(sorted_feat_names)), sorted_importance)
    plt.yticks(range(len(sorted_feat_names)), sorted_feat_names)
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()


# ====== 10. Train & Evaluate ======
for name, model in models.items():

    print("\n" + "="*30)
    print(f"Training: {name}")
    print("="*30)

    clf = Pipeline([
        ("preprocessor", preprocess),
        ("model", model)
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=3))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


    # ===== Feature Importance =====
    if name in ["XGBoost", "Random Forest", "Logistic Regression"]:

        # SAFEST way to get exact feature order
        feature_names = clf.named_steps["preprocessor"].get_feature_names_out()

        if name in ["XGBoost", "Random Forest"]:
            importance = clf.named_steps["model"].feature_importances_

        elif name == "Logistic Regression":
            importance = np.mean(
                np.abs(clf.named_steps["model"].coef_), axis=0
            )

        plot_feature_importance(
            feature_names,
            importance,
            title=f"{name} Feature Importance",
            filename=f"feature_importance_plots_main/{name}_feature_importance.png"
        )

        # ===== Permutation Importance =====
        perm_importance = permutation_importance(
            clf, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1
        )

        print("\nTop 10 Features by Permutation Importance:")
        top_idx = perm_importance.importances_mean.argsort()[::-1][:10]
        for i in top_idx:
            print(f"{feature_names[i]}: {perm_importance.importances_mean[i]:.4f}")


Training: XGBoost

Classification Report:
              precision    recall  f1-score   support

           0      0.972     0.758     0.852     37000
           1      0.987     0.819     0.895     22960
           2      0.606     0.929     0.733     12137
           3      0.435     0.672     0.528     10235

    accuracy                          0.790     82332
   macro avg      0.750     0.795     0.752     82332
weighted avg      0.855     0.790     0.806     82332


Confusion Matrix:
[[28042    27   850  8081]
 [   29 18808  3900   223]
 [   96   148 11277   616]
 [  684    77  2597  6877]]

Top 10 Features by Permutation Importance:
num__dload: 0.1028
num__dpkts: 0.0685
num__response_body_len: 0.0481
num__sttl: 0.0461
num__ct_flw_http_mthd: 0.0265
num__dmean: 0.0165
num__dttl: 0.0140
num__ct_ftp_cmd: 0.0119
cat__proto_a/n: 0.0117
num__ct_dst_src_ltm: 0.0108

Training: Random Forest

Classification Report:
              precision    recall  f1-score   support

           0     

c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Top 10 Features by Permutation Importance:
num__dpkts: 0.0783
num__ct_srv_src: 0.0106
num__ct_flw_http_mthd: 0.0083
cat__proto_a/n: 0.0080
num__response_body_len: 0.0070
num__sttl: 0.0068
num__ct_src_dport_ltm: 0.0053
num__sjit: 0.0051
num__dttl: 0.0051
num__sinpkt: 0.0037

Training: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0      0.962     0.595     0.735     37000
           1      0.907     0.808     0.854     22960
           2      0.546     0.702     0.614     12137
           3      0.323     0.737     0.449     10235

    accuracy                          0.688     82332
   macro avg      0.684     0.710     0.663     82332
weighted avg      0.806     0.688     0.715     82332


Confusion Matrix:
[[22007    30  1890 13073]
 [  111 18541  3613   695]
 [  585   956  8526  2070]
 [  180   916  1596  7543]]

Top 10 Features by Permutation Importance:
num__dpkts: 0.1733
num__dwin: 0.1534
num__sloss: 0.1021
num__sb

In [8]:
import matplotlib.pyplot as plt
import numpy as np
import os

# ── Permutation Importance data ────────────────────────────────────────────
model_data = {
    'XGBoost': {
        'features': ['dload', 'dpkts', 'response_body_len', 'sttl',
                     'ct_flw_http_mthd', 'dmean', 'dttl', 'ct_ftp_cmd',
                     'proto_a/n', 'ct_dst_src_ltm'],
        'values':   [0.1028, 0.0685, 0.0481, 0.0461,
                     0.0265, 0.0165, 0.0140, 0.0119,
                     0.0117, 0.0108],
        'color': '#6C63FF'
    },
    'Random Forest': {
        'features': ['dpkts', 'ct_srv_src', 'ct_flw_http_mthd', 'proto_a/n',
                     'response_body_len', 'sttl', 'ct_src_dport_ltm', 'sjit',
                     'dttl', 'sinpkt'],
        'values':   [0.0783, 0.0106, 0.0083, 0.0080,
                     0.0070, 0.0068, 0.0053, 0.0051,
                     0.0051, 0.0037],
        'color': '#00C9A7'
    },
    'Logistic Regression': {
        'features': ['dpkts', 'dwin', 'sloss', 'sbytes',
                     'ackdat', 'is_ftp_login', 'sinpkt', 'spkts',
                     'ct_srv_src', 'ct_ftp_cmd'],
        'values':   [0.1733, 0.1534, 0.1021, 0.0903,
                     0.0901, 0.0584, 0.0322, 0.0268,
                     0.0233, 0.0213],
        'color': '#FF6B6B'
    }
}

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 8))
fig.patch.set_facecolor('#0F1117')

fig.suptitle(
    'Top 10 Features by Permutation Importance',
    fontsize=18, fontweight='bold', color='white', y=1.02
)

for ax, (model_name, info) in zip(axes, model_data.items()):

    features = info['features'][::-1]
    values   = info['values'][::-1]
    color    = info['color']
    y_pos    = np.arange(len(features))

    ax.set_facecolor('#1A1D2E')

    # Bars with opacity gradient
    for i, (val, alpha) in enumerate(
        zip(values, [0.35 + 0.65 * (i / (len(values) - 1)) for i in range(len(values))])
    ):
        ax.barh(y_pos[i], val, color=color, alpha=alpha, edgecolor='none', height=0.65)

    # Value labels
    max_val = max(values)
    for i, val in enumerate(values):
        ax.text(val + max_val * 0.013, y_pos[i],
                f'{val:.4f}',
                va='center', ha='left',
                fontsize=9, color='#CCCCCC', fontweight='bold')

    # Styling
    ax.set_yticks(y_pos)
    ax.set_yticklabels(features, fontsize=10, color='white')
    ax.set_xlabel('Permutation Importance Score', fontsize=10, color='#AAAAAA')
    ax.tick_params(axis='x', colors='#AAAAAA', labelsize=9)
    ax.tick_params(axis='y', colors='white', length=0)
    ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    ax.xaxis.grid(True, color='#2C2F45', linewidth=0.7, linestyle='--')
    ax.set_axisbelow(True)
    ax.set_xlim(0, max_val * 1.32)
    ax.set_title(model_name, fontsize=13, fontweight='bold', color=color, pad=10)
    ax.axvline(0, color=color, linewidth=2.5, alpha=0.7)

os.makedirs('feature_importance_plots_main', exist_ok=True)
plt.tight_layout(pad=2.5)
plt.savefig('feature_importance_plots_main/permutation_importance_3models.png',
            dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✓ Saved → feature_importance_plots_main/permutation_importance_3models.png')


✓ Saved → feature_importance_plots_main/permutation_importance_3models.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13280\698518785.py:85: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
